In [ ]:
import requests
import pandas as pd
import time
import os

# ===== Cell 2: Data load =====

# --- 1. Configuration ---
TIINGO_API_KEY = "5cbffd02e11af8771393bbeb57a4b3a90de96f8e"  # Replace with your actual key
SAVE_DIR = "../data/cnn_lstm_data"
os.makedirs(SAVE_DIR, exist_ok=True)

# Period: 2017 through 2022
START_DATE = "2017-01-01"
END_DATE = "2022-12-31"

## TEST ON AAPL

# stocks_list = ["AAPL"]  # TEST before running the next 49 stocks

## END OF TEST

stocks_list = [
    # --- 13 NASDAQ-100 Leaders ---
    "AMZN", "NVDA", "GOOGL", "META", "TSLA", "AVGO", "PEP", "COST", "ADBE", "AMD", "NFLX", "QCOM", "INTC",
    # --- 7 Energy ---
    "XOM", "CVX", "SHEL", "COP", "TTE", "SLB", "BP",
    # --- 7 Materials ---
    "LIN", "BHP", "RIO", "SHW", "SCCO", "FCX", "NEM",
    # --- 7 Health Care ---
    "LLY", "JNJ", "UNH", "MRK", "ABBV", "PFE", "TMO",
    # --- 7 Financial ---
    "JPM", "BAC", "WFC", "MS", "GS", "BLK", "SCHW",
    # --- 7 Real Estate ---
    "PLD", "AMT", "EQIX", "WELL", "SPG", "DLR", "O"
]


def fetch_and_save_raw(ticker):
    """
    Fetches adjusted OHLCV data and saves to a clearly labeled CSV.
    """
    headers = {'Content-Type': 'application/json'}
    url = f"https://api.tiingo.com/tiingo/daily/{ticker}/prices"
    params = {
        'startDate': START_DATE,
        'endDate': END_DATE,
        'token': TIINGO_API_KEY,
        'format': 'json'
    }

    try:
        response = requests.get(url, params=params, headers=headers)
        if response.status_code == 200:
            raw_data = response.json()
            if not raw_data:
                print(f"⚠️ No data found for {ticker}")
                return False

            df = pd.DataFrame(raw_data)

            # --- Data Selection & Labeling ---
            # We explicitly select 'adj' columns to handle splits/dividends
            # and rename them for human-readability.
            clean_df = df[[
                'date', 'adjOpen', 'adjHigh', 'adjLow', 'adjClose', 'adjVolume'
            ]].copy()

            clean_df.columns = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
            clean_df['Ticker'] = ticker # For easier identification later

            # Format date for readability (e.g., 2017-01-03)
            clean_df['Date'] = pd.to_datetime(clean_df['Date']).dt.date

            # Save locally
            file_path = os.path.join(SAVE_DIR, f"{ticker}_RAW.csv")
            clean_df.to_csv(file_path, index=False)
            print(f"✅ {ticker}: {len(clean_df)} days saved.")
            return True
        else:
            print(f"❌ Error {response.status_code} for {ticker}: {response.text}")
            return False
    except Exception as e:
        print(f"⚠️ Exception fetching {ticker}: {e}")
        return False

# --- 2. Execution Loop with Rate Limiting ---
print(f"🚀 Starting collection for {len(stocks_list)} stocks...")
start_time = time.time()

for i, ticker in enumerate(stocks_list):
    success = fetch_and_save_raw(ticker)

    # Tiingo Free Tier: 50 requests/hour (1 every 72s)
    # We sleep 75s to be safe against network jitter
    if i < len(stocks_list) - 1:
        print(f"[{i+1}/{len(stocks_list)}] Sleeping 5s to respect API limits...")
        time.sleep(5)

total_time = (time.time() - start_time) / 60
print(f"🏁 Collection Finished! Total time: {total_time:.2f} minutes.")

# ===== End of Cell 2 =====

In [2]:
import pandas as pd
import numpy as np
import torch
import os
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

# --- 1. Path Configuration ---
# Moving from Project/Notebooks/scripts/ up to Project root, then to data/
BASE_DATA_DIR = "../../data/cnn_lstm_data/raw_data"
RAW_DIR = os.path.join(BASE_DATA_DIR)
PROCESSED_DIR = os.path.join(BASE_DATA_DIR, "processed_data")

# Create the processed directory if it doesn't exist
os.makedirs(PROCESSED_DIR, exist_ok=True)

MASTER_SAVE_PATH = os.path.join(PROCESSED_DIR, "master_dataset_M60.pt")

# --- 2. Hyperparameters ---
M = 60              # Lookback window
ATR_MA_PERIOD = 14  # Period for technical indicators
ATR_MULTIPLIER = 0.5

def calculate_indicators(df):
    """Computes ATR% and MA% Distance."""
    # Moving Average (14-day)
    df['MA'] = df['Close'].rolling(window=ATR_MA_PERIOD).mean()
    # MA Distance %: (Price - MA) / MA
    df['MA_Dist_Pct'] = (df['Close'] - df['MA']) / df['MA']

    # ATR (14-day)
    high_low = df['High'] - df['Low']
    high_close = (df['High'] - df['Close'].shift()).abs()
    low_close = (df['Low'] - df['Close'].shift()).abs()
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    df['ATR'] = tr.rolling(window=ATR_MA_PERIOD).mean()
    # ATR %: ATR / Close
    df['ATR_Pct'] = df['ATR'] / df['Close']

    return df.dropna(subset=['MA_Dist_Pct', 'ATR_Pct']).copy()

def process_and_normalize():
    all_raw_windows = []
    files = [f for f in os.listdir(RAW_DIR) if f.endswith('_RAW.csv')]

    if not files:
        print(f"❌ No CSV files found in {RAW_DIR}. Check your data collection step!")
        return

    print(f"🛠️ Step 1: Feature Engineering & Windowing {len(files)} stocks...")

    for file in tqdm(files):
        df = pd.read_csv(os.path.join(RAW_DIR, file))

        # A. Calculate Technical Features
        df = calculate_indicators(df)

        # B. Define Target Labels (T+1 move relative to ATR)
        df['Next_Close'] = df['Close'].shift(-1)
        df['Move'] = df['Next_Close'] - df['Close']

        df['Label'] = 1 # Neutral
        df.loc[df['Move'] > (df['ATR'] * ATR_MULTIPLIER), 'Label'] = 2 # Buy
        df.loc[df['Move'] < -(df['ATR'] * ATR_MULTIPLIER), 'Label'] = 0 # Sell

        df.dropna(subset=['Next_Close'], inplace=True)

        # C. Feature Set (7 dimensions)
        feature_cols = ['Open', 'High', 'Low', 'Close', 'Volume', 'MA_Dist_Pct', 'ATR_Pct']
        data_matrix = df[feature_cols].values
        labels = df['Label'].values
        dates = df['Date'].values

        # D. Window Generation
        for i in range(len(df) - M):
            x_win = data_matrix[i : i + M]
            y_lab = labels[i + M - 1]
            win_date = dates[i + M - 1]

            all_raw_windows.append({
                'x_raw': x_win,
                'y': y_lab,
                'date': win_date
            })

    # --- 2. Global Normalization ---
    print(f"⚖️ Step 2: Globally scaling {len(all_raw_windows)} samples...")
    X_all = np.array([s['x_raw'] for s in all_raw_windows])
    N, seq_len, feat_dim = X_all.shape

    scaler = StandardScaler()
    X_flattened = X_all.reshape(-1, feat_dim)
    X_scaled = scaler.fit_transform(X_flattened).reshape(N, seq_len, feat_dim)

    # --- 3. Save Master File (Chronologically Sorted) ---
    master_data = []
    y_counts = {0: 0, 1: 0, 2: 0}

    for i in range(len(all_raw_windows)):
        label = int(all_raw_windows[i]['y'])
        y_counts[label] += 1

        master_data.append({
            'x': X_scaled[i],
            'y': label,
            'date': all_raw_windows[i]['date']
        })

    master_data.sort(key=lambda x: x['date'])

    torch.save(master_data, MASTER_SAVE_PATH)

    # --- 4. Final Stats ---
    print(f"\n✅ Master dataset ready: {len(master_data)} samples.")
    print(f"📍 Location: {MASTER_SAVE_PATH}")
    print("\n📊 --- Class Balance ---")
    total = len(master_data)
    print(f"Sell (0): {y_counts[0]} ({y_counts[0]/total:.1%})")
    print(f"Neutral (1): {y_counts[1]} ({y_counts[1]/total:.1%})")
    print(f"Buy (2): {y_counts[2]} ({y_counts[2]/total:.1%})")

if __name__ == "__main__":
    process_and_normalize()

🛠️ Step 1: Feature Engineering & Windowing 50 stocks...


100%|██████████| 50/50 [00:00<00:00, 78.04it/s]


⚖️ Step 2: Globally scaling 71355 samples...

✅ Master dataset ready: 71355 samples.
📍 Location: ../../data/cnn_lstm_data/raw_data\processed_data\master_dataset_M60.pt

📊 --- Class Balance ---
Sell (0): 14368 (20.1%)
Neutral (1): 40203 (56.3%)
Buy (2): 16784 (23.5%)
